# Probability Utilities

Shared probability functions used across quant research projects.
Each function is derived from first principles and demonstrated with concrete examples.

| Function | What it computes |
|----------|------------------|
| `expected_value` | Weighted average payoff across outcomes |
| `combinations_count` | C(n,k) — ways to choose k from n |
| `bayes` | Posterior probability from Bayes' theorem |
| `normal_pdf` | Probability density of a normal distribution |

---

## 1. Expected Value

$$EV = \sum_{i} p_i \cdot v_i$$

Takes a list of `(probability, payoff)` pairs and returns the probability-weighted average outcome.
This is the foundation of every rational decision under uncertainty.

In [ ]:
import math


def expected_value(outcomes):
    """
    Calculate expected value from a list of (probability, payoff) tuples.

    Args:
        outcomes: list of (probability, payoff) tuples

    Returns:
        float: expected value
    """
    return sum(p * v for p, v in outcomes)

In [ ]:
# Example 1: simple coin flip bet
coin_flip = [(0.5, 10), (0.5, -5)]
print(f"Coin flip EV: ${expected_value(coin_flip)}")

# Example 2: dice game — win $15 on 6, lose $3 otherwise
dice_game = [(1/6, 15), (5/6, -3)]
print(f"Dice game EV: ${expected_value(dice_game):.4f}")

# Example 3: three-outcome trade
trade = [
    (0.40, 800),   # large win
    (0.35, 200),   # small win
    (0.25, -600),  # loss
]
print(f"Trade EV: ${expected_value(trade):.2f}")

## 2. Combinations C(n, k)

$$C(n, k) = \binom{n}{k} = \frac{n!}{k!(n-k)!}$$

Counts the number of ways to choose $k$ items from $n$ **without replacement** and **without regard to order**.

Used heavily in probability to count favorable outcomes (e.g., how many ways to get exactly 2 heads in 5 flips).

In [ ]:
def combinations_count(n, k):
    """C(n, k) — number of ways to choose k items from n without replacement."""
    return math.comb(n, k)


# Build Pascal's triangle to 6 rows
print("Pascal's Triangle (C(n,k) values):\n")
for n in range(7):
    row = [combinations_count(n, k) for k in range(n + 1)]
    print(f"n={n}: {row}")

print()

# Applied: probability of exactly k heads in 10 flips
n_flips = 10
total = 2 ** n_flips
print(f"Probability of exactly k heads in {n_flips} fair coin flips:")
for k in range(n_flips + 1):
    ways = combinations_count(n_flips, k)
    prob = ways / total
    bar = '█' * int(prob * 100)
    print(f"  k={k:>2}: {prob:.4f}  {bar}")

## 3. Bayes' Theorem

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

The general form — pass in the three components directly.

- **prior** $P(A)$: what you believe before the evidence
- **likelihood** $P(B|A)$: how probable the evidence is if $A$ is true
- **marginal** $P(B)$: total probability of the evidence under all hypotheses

In [ ]:
def bayes(prior, likelihood, marginal):
    """
    Bayes' theorem: P(A|B) = P(B|A) * P(A) / P(B)

    Args:
        prior: P(A)
        likelihood: P(B|A)
        marginal: P(B)

    Returns:
        float: posterior P(A|B)
    """
    return (likelihood * prior) / marginal


# Classic example: medical test
# Disease prevalence: 1% of population
# Test sensitivity (true positive rate): 99%
# Test specificity (true negative rate): 95% → false positive rate 5%

p_disease   = 0.01          # prior: 1% base rate
p_pos_given_disease   = 0.99  # sensitivity
p_pos_given_no_disease = 0.05  # false positive rate

# Marginal: total probability of a positive test
p_positive = p_pos_given_disease * p_disease + p_pos_given_no_disease * (1 - p_disease)

posterior = bayes(p_disease, p_pos_given_disease, p_positive)

print("Medical Test Example")
print(f"  Base rate of disease:       {p_disease*100:.0f}%")
print(f"  Test sensitivity:           {p_pos_given_disease*100:.0f}%")
print(f"  False positive rate:        {p_pos_given_no_disease*100:.0f}%")
print(f"  P(positive test):           {p_positive*100:.2f}%")
print(f"  P(disease | positive test): {posterior*100:.2f}%")
print()
print("Counterintuitive: even with a 99% accurate test, a positive result")
print("only means ~16.7% chance of disease when the base rate is 1%.")

## 4. Normal PDF

The **probability density function** of the normal distribution:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2\right)$$

Used to model returns, price changes, and many financial variables.
Note: $f(x)$ is a **density**, not a probability — it can exceed 1.

In [ ]:
def normal_pdf(x, mu=0, sigma=1):
    """Normal distribution probability density function."""
    return (1 / (sigma * math.sqrt(2 * math.pi))) * math.exp(-0.5 * ((x - mu) / sigma) ** 2)


# Visualize the standard normal as ASCII
print("Standard Normal PDF (mu=0, sigma=1):\n")
xs = [x / 10 for x in range(-30, 31)]
peak = normal_pdf(0)
for x in xs:
    density = normal_pdf(x)
    bar = '█' * int(density / peak * 40)
    print(f"  x={x:+.1f}: {density:.4f}  {bar}")

print()

# Market returns example: daily returns ~ N(0.0005, 0.012)
mu_ret, sigma_ret = 0.0005, 0.012
print(f"Daily returns modeled as N(mu={mu_ret}, sigma={sigma_ret})")
for ret in [-0.03, -0.02, -0.01, 0, 0.01, 0.02, 0.03]:
    density = normal_pdf(ret, mu_ret, sigma_ret)
    print(f"  return={ret:+.2f}: density={density:.2f}")

---
## Summary

These four functions are the building blocks behind most quantitative probability work:

- **EV** — the workhorse for any bet/trade evaluation
- **C(n,k)** — essential for combinatorics (options pricing, probability counting)
- **Bayes** — how to reason with partial information
- **Normal PDF** — foundation for modeling continuous financial variables